In [157]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [158]:
output_dir = Path("../output")
datafiles = sorted(output_dir.glob("*/*/experiment_data.*"))
print(f"Found {len(datafiles)} data files:\n")
for f in datafiles:
    task_dir = f.parent.parent.name
    run_dir = f.parent.name
    size_kb = f.stat().st_size / 1024
    print(f"  {task_dir}/{run_dir}/{f.name}  ({size_kb:.1f} KB)")

Found 576 data files:

  stop_signal_task_(rdoc)/2026-03-03_16-49-51/experiment_data.json  (280.0 KB)
  stop_signal_task_(rdoc)/2026-03-03_17-00-24/experiment_data.json  (293.5 KB)
  stop_signal_task_(rdoc)/2026-03-03_17-00-26/experiment_data.json  (265.9 KB)
  stop_signal_task_(rdoc)/2026-03-03_17-00-28/experiment_data.json  (266.3 KB)
  stop_signal_task_(rdoc)/2026-03-03_17-00-30/experiment_data.json  (294.3 KB)
  stop_signal_task_(rdoc)/2026-03-03_17-00-32/experiment_data.json  (266.2 KB)
  stop_signal_task_(rdoc)/2026-03-03_17-00-34/experiment_data.json  (266.3 KB)
  stop_signal_task_(rdoc)/2026-03-03_17-00-36/experiment_data.json  (266.1 KB)
  stop_signal_task_(rdoc)/2026-03-03_17-00-38/experiment_data.json  (266.1 KB)
  stop_signal_task_(rdoc)/2026-03-03_17-00-40/experiment_data.json  (266.5 KB)
  stop_signal_task_(rdoc)/2026-03-03_17-00-42/experiment_data.json  (266.1 KB)
  stop_signal_task_(rdoc)/2026-03-03_17-00-44/experiment_data.json  (293.7 KB)
  stop_signal_task_(rdoc)/202

In [159]:
def detect_format(df: pd.DataFrame) -> str:
    """Detect experiment data format from column names."""
    cols = set(df.columns)
    if "signal" in cols and "SSD" in cols:
        return "stopit"
    if "trial_id" in cols and "SS_trial_type" in cols:
        return "expfactory_stop_signal"
    if "trial_id" in cols and "stim_color" in cols:
        return "expfactory_stroop"
    if "text" in cols and "colour" in cols:
        return "cognitionrun_stroop"
    if "trial_id" in cols:
        return "expfactory_generic"
    return "unknown"


def analyze_expfactory_stroop(df: pd.DataFrame) -> dict:
    """Analyze ExpFactory RDoC Stroop data."""
    test = df[df["trial_id"] == "test_trial"].copy()
    test["rt"] = pd.to_numeric(test["rt"], errors="coerce")
    test["correct_trial"] = pd.to_numeric(test["correct_trial"], errors="coerce")

    results = {"task": "Stroop (ExpFactory)", "format": "expfactory_stroop", "n_total": len(test)}
    summary = {}
    for cond in ["congruent", "incongruent"]:
        g = test[test["condition"] == cond]
        responded = g[g["rt"].notna()]
        correct = g[g["correct_trial"] == 1]
        rt_correct = correct["rt"].dropna()
        summary[f"{cond}_accuracy"] = g["correct_trial"].mean() if len(g) else np.nan
        summary[f"{cond}_omission_rate"] = g["rt"].isna().mean() if len(g) else np.nan
        summary[f"{cond}_rt"] = rt_correct.mean() if len(rt_correct) else np.nan
    results["summary"] = summary
    return results


def analyze_expfactory_stop_signal(df: pd.DataFrame) -> dict:
    """Analyze ExpFactory RDoC Stop Signal data."""
    test = df[df["trial_id"] == "test_trial"].copy()
    test["rt"] = pd.to_numeric(test["rt"], errors="coerce")
    test["correct_trial"] = pd.to_numeric(test["correct_trial"], errors="coerce")
    test["SSD"] = pd.to_numeric(test.get("SSD", pd.Series(dtype=float)), errors="coerce")

    go = test[test["condition"] == "go"]
    stop = test[test["condition"] == "stop"]

    go_correct_rt = go[(go["correct_trial"] == 1) & go["rt"].notna()]["rt"]
    go_all_responded_rt = go[go["rt"].notna()]["rt"]
    stop_failed = stop[(stop["correct_trial"] == 0) & stop["rt"].notna()]
    stop_ssd = stop["SSD"].dropna()

    results = {
        "task": "Stop Signal (ExpFactory)",
        "format": "expfactory_stop_signal",
        "n_total": len(test),
        "summary": {
            "go_accuracy": go["correct_trial"].mean() if len(go) else np.nan,
            "go_omission_rate": go["rt"].isna().mean() if len(go) else np.nan,
            "go_rt": go_correct_rt.mean() if len(go_correct_rt) else np.nan,
            "go_rt_all_responses": go_all_responded_rt.mean() if len(go_all_responded_rt) else np.nan,
            "mean_stop_failure_RT": stop_failed["rt"].mean() if len(stop_failed) else np.nan,
            "stop_accuracy": stop["correct_trial"].mean() if len(stop) else np.nan,
            "min_SSD": stop_ssd.min() if len(stop_ssd) else np.nan,
            "mean_SSD": stop_ssd.mean() if len(stop_ssd) else np.nan,
            "max_SSD": stop_ssd.max() if len(stop_ssd) else np.nan,
            "final_SSD": stop["SSD"].iloc[-1] if len(stop) else np.nan,
        },
    }
    return results


def analyze_cognitionrun_stroop(df: pd.DataFrame) -> dict:
    """Analyze Cognition.run Stroop data."""
    trials = df[(df["text"].notna()) & (df["text"] != "") & (df["colour"].notna()) & (df["colour"] != "")].copy()
    trials["rt"] = pd.to_numeric(trials["rt"], errors="coerce")
    trials["condition"] = np.where(trials["text"] == trials["colour"], "congruent", "incongruent")
    key_map = {"red": "r", "green": "g", "blue": "b", "yellow": "y"}
    trials["correct_key"] = trials["colour"].map(key_map)
    trials["correct"] = trials["response"] == trials["correct_key"]
    trials["omission"] = trials["response"].isna() | (trials["rt"].isna())

    results = {"task": "Stroop (Cognition.run)", "format": "cognitionrun_stroop", "n_total": len(trials)}
    summary = {}
    for cond in ["congruent", "incongruent"]:
        g = trials[trials["condition"] == cond]
        correct = g[g["correct"]]
        rt_correct = correct["rt"].dropna()
        summary[f"{cond}_accuracy"] = g["correct"].mean() if len(g) else np.nan
        summary[f"{cond}_omission_rate"] = g["omission"].mean() if len(g) else np.nan
        summary[f"{cond}_rt"] = rt_correct.mean() if len(rt_correct) else np.nan
    results["summary"] = summary
    return results


def analyze_stopit(df: pd.DataFrame) -> dict:
    """Analyze STOP-IT Stop Signal data."""
    df = df.copy()
    df["rt"] = pd.to_numeric(df["rt"], errors="coerce")
    df["correct"] = df["correct"].astype(str).str.lower() == "true"
    df["block_i"] = pd.to_numeric(df["block_i"], errors="coerce")
    df["SSD"] = pd.to_numeric(df["SSD"], errors="coerce")

    exp = df[df["block_i"] > 0]  # exclude practice block 0
    go = exp[exp["signal"] == "no"]
    stop = exp[exp["signal"] == "yes"]

    go_omission = go["rt"].isna() | (go["response"] == "undefined")
    go_correct_rt = go[go["correct"] & ~go_omission]["rt"].dropna()
    go_all_responded_rt = go[~go_omission]["rt"].dropna()
    stop_failed = stop[~stop["correct"]]
    stop_failed_rt = stop_failed["rt"].dropna()
    stop_ssd = stop["SSD"].dropna()

    results = {
        "task": "Stop Signal (STOP-IT)",
        "format": "stopit",
        "n_total": len(exp),
        "summary": {
            "go_accuracy": go["correct"].mean() if len(go) else np.nan,
            "go_omission_rate": go_omission.mean() if len(go) else np.nan,
            "go_rt": go_correct_rt.mean() if len(go_correct_rt) else np.nan,
            "go_rt_all_responses": go_all_responded_rt.mean() if len(go_all_responded_rt) else np.nan,
            "mean_stop_failure_RT": stop_failed_rt.mean() if len(stop_failed_rt) else np.nan,
            "stop_accuracy": stop["correct"].mean() if len(stop) else np.nan,
            "min_SSD": stop_ssd.min() if len(stop_ssd) else np.nan,
            "mean_SSD": stop_ssd.mean() if len(stop_ssd) else np.nan,
            "max_SSD": stop_ssd.max() if len(stop_ssd) else np.nan,
            "final_SSD": stop["SSD"].iloc[-1] if len(stop) else np.nan,
        },
    }
    return results


ANALYZERS = {
    "expfactory_stroop": analyze_expfactory_stroop,
    "expfactory_stop_signal": analyze_expfactory_stop_signal,
    "cognitionrun_stroop": analyze_cognitionrun_stroop,
    "stopit": analyze_stopit,
}

print("Defined analyzers for:", list(ANALYZERS.keys()))

Defined analyzers for: ['expfactory_stroop', 'expfactory_stop_signal', 'cognitionrun_stroop', 'stopit']


In [160]:
# Analyze all data files
all_results = []
for f in datafiles:
    df = pd.read_csv(f)
    fmt = detect_format(df)
    label = f"{f.parent.parent.name}/{f.parent.name}"

    if fmt in ANALYZERS:
        results = ANALYZERS[fmt](df)
        results["file"] = label
        all_results.append(results)
    else:
        print(f"--- {label}: format={fmt} (no analyzer), {len(df)} rows ---")

print(f"Analyzed {len(all_results)} files")

--- stop_signal_task_(rdoc)/2026-03-03_16-49-51: format=unknown (no analyzer), 0 rows ---
--- stop_signal_task_(rdoc)/2026-03-03_17-00-24: format=unknown (no analyzer), 0 rows ---
--- stop_signal_task_(rdoc)/2026-03-03_17-00-26: format=unknown (no analyzer), 0 rows ---
--- stop_signal_task_(rdoc)/2026-03-03_17-00-28: format=unknown (no analyzer), 0 rows ---
--- stop_signal_task_(rdoc)/2026-03-03_17-00-30: format=unknown (no analyzer), 0 rows ---
--- stop_signal_task_(rdoc)/2026-03-03_17-00-32: format=unknown (no analyzer), 0 rows ---
--- stop_signal_task_(rdoc)/2026-03-03_17-00-34: format=unknown (no analyzer), 0 rows ---
--- stop_signal_task_(rdoc)/2026-03-03_17-00-36: format=unknown (no analyzer), 0 rows ---
--- stop_signal_task_(rdoc)/2026-03-03_17-00-38: format=unknown (no analyzer), 0 rows ---
--- stop_signal_task_(rdoc)/2026-03-03_17-00-40: format=unknown (no analyzer), 0 rows ---
--- stop_signal_task_(rdoc)/2026-03-03_17-00-42: format=unknown (no analyzer), 0 rows ---
--- stop_s

In [161]:
STOP_SIGNAL_COLS = [
    "task", "run", "go_accuracy", "go_omission_rate", "go_rt", "go_rt_all_responses",
    "mean_stop_failure_RT", "stop_accuracy", "max_SSD", "mean_SSD", "min_SSD", "final_SSD",
]
STROOP_COLS = [
    "task", "run", "congruent_accuracy", "congruent_omission_rate", "congruent_rt",
    "incongruent_accuracy", "incongruent_omission_rate", "incongruent_rt",
]

stop_rows, stroop_rows = [], []
for r in all_results:
    row = {"task": r["task"], "run": r["file"].split("/")[-1], **r["summary"]}
    if r["format"] in ("expfactory_stop_signal", "stopit"):
        stop_rows.append(row)
    else:
        stroop_rows.append(row)

if stop_rows:
    stop_df = pd.DataFrame(stop_rows, columns=STOP_SIGNAL_COLS)
    print("Stop Signal Summary\n")
    print(stop_df.to_string(index=False, float_format=lambda x: f"{x:.3f}" if abs(x) < 2 else f"{x:.1f}"))
    stop_df.to_csv(output_dir / "stop_signal_summary.csv", index=False)
    print(f"\nSaved to {output_dir / 'stop_signal_summary.csv'}")

if stroop_rows:
    stroop_df = pd.DataFrame(stroop_rows, columns=STROOP_COLS)
    print("\n\nStroop Summary\n")
    print(stroop_df.to_string(index=False, float_format=lambda x: f"{x:.3f}" if abs(x) < 2 else f"{x:.1f}"))
    stroop_df.to_csv(output_dir / "stroop_summary.csv", index=False)
    print(f"\nSaved to {output_dir / 'stroop_summary.csv'}")

Stop Signal Summary

                 task                 run  go_accuracy  go_omission_rate  go_rt  go_rt_all_responses  mean_stop_failure_RT  stop_accuracy  max_SSD  mean_SSD  min_SSD  final_SSD
Stop Signal (STOP-IT) 2026-03-03_16-49-53        0.839             0.141  598.6                599.2                 561.8          0.516      750     481.2      150        250
Stop Signal (STOP-IT) 2026-03-03_17-01-44        0.865             0.109  516.4                518.4                 465.8          0.484      600     312.5       50        250
Stop Signal (STOP-IT) 2026-03-03_17-01-46        0.932             0.047  531.7                531.3                 453.0          0.547      600     393.8      200        550
Stop Signal (STOP-IT) 2026-03-03_17-01-48        0.922             0.068  612.6                611.0                 530.3          0.547      700     459.4      250        550
Stop Signal (STOP-IT) 2026-03-03_17-01-50        0.896             0.062  607.2               

In [162]:
# ── Sequential Effects & RT Distribution Analysis ──────────────────────
# Analyze trial-level data for humanlike temporal structure:
#   - Post-error slowing (PES)
#   - Post-stop-signal slowing (PSSS)
#   - Lag-1 autocorrelation
#   - RT distribution shape (skewness)
#   - Fatigue drift (RT ~ trial number)

import json
from scipy import stats as sp_stats


def load_trial_data(task_dir: str) -> list[tuple[str, pd.DataFrame, list[dict]]]:
    """Load paired experiment_data + bot_log for each run in a task directory."""
    runs = []
    task_path = output_dir / task_dir
    if not task_path.exists():
        return runs
    for run_dir in sorted(task_path.iterdir()):
        exp_files = list(run_dir.glob("experiment_data.*"))
        bot_file = run_dir / "bot_log.json"
        if not exp_files or not bot_file.exists():
            continue
        try:
            exp_file = exp_files[0]
            if exp_file.suffix == ".json":
                exp_df = pd.read_json(exp_file)
            else:
                exp_df = pd.read_csv(exp_file)
            with open(bot_file) as f:
                bot_log = json.load(f)
            if isinstance(bot_log, dict):
                bot_log = bot_log.get("trials", [])
            if len(bot_log) < 10:
                continue
            runs.append((run_dir.name, exp_df, bot_log))
        except Exception:
            continue
    return runs


def sequential_analysis_stop_signal(runs: list, fmt: str) -> dict:
    """Compute sequential effects for stop signal data.

    Returns dict with PES, post-stop slowing, lag-1 autocorrelation,
    skewness, and drift metrics aggregated across runs.
    """
    pes_list, psss_list, psss_success_list, psss_failure_list = [], [], [], []
    lag1_list, skew_list, drift_list = [], [], []

    for run_name, exp_df, bot_log in runs:
        # Build trial-level DataFrame from experiment data
        if fmt == "expfactory_stop_signal":
            if "trial_id" not in exp_df.columns:
                continue
            test = exp_df[exp_df["trial_id"] == "test_trial"].copy().reset_index(drop=True)
            test["rt"] = pd.to_numeric(test["rt"], errors="coerce")
            test["correct_trial"] = pd.to_numeric(test["correct_trial"], errors="coerce")
            is_go = test["condition"] == "go"
            is_stop = test["condition"] == "stop"
        elif fmt == "stopit":
            test = exp_df.copy()
            test["rt"] = pd.to_numeric(test["rt"], errors="coerce")
            test["correct"] = test["correct"].astype(str).str.lower() == "true"
            test["block_i"] = pd.to_numeric(test["block_i"], errors="coerce")
            test = test[test["block_i"] > 0].reset_index(drop=True)
            is_go = test["signal"] == "no"
            is_stop = test["signal"] == "yes"
            test["correct_trial"] = test["correct"].astype(float)
        else:
            continue

        go_rts = test.loc[is_go, "rt"].values
        valid_go_rts = go_rts[~np.isnan(go_rts)]
        if len(valid_go_rts) < 20:
            continue

        # ── Post-error slowing ──
        prev_correct = test["correct_trial"].shift(1)
        go_after_correct = test[is_go & (prev_correct == 1)]["rt"].dropna()
        go_after_error = test[is_go & (prev_correct == 0)]["rt"].dropna()
        if len(go_after_correct) > 5 and len(go_after_error) > 2:
            pes_list.append(go_after_error.mean() - go_after_correct.mean())

        # ── Post-stop-signal slowing ──
        prev_is_stop = is_stop.shift(1).fillna(False)
        prev_is_go = is_go.shift(1).fillna(False)
        go_after_go = test[is_go & prev_is_go]["rt"].dropna()
        go_after_stop = test[is_go & prev_is_stop]["rt"].dropna()
        if len(go_after_go) > 5 and len(go_after_stop) > 2:
            psss_list.append(go_after_stop.mean() - go_after_go.mean())

        # Split by successful vs failed inhibition
        prev_correct_val = test["correct_trial"].shift(1)
        go_after_succ_stop = test[is_go & prev_is_stop & (prev_correct_val == 1)]["rt"].dropna()
        go_after_fail_stop = test[is_go & prev_is_stop & (prev_correct_val == 0)]["rt"].dropna()
        if len(go_after_succ_stop) > 2:
            psss_success_list.append(go_after_succ_stop.mean() - go_after_go.mean())
        if len(go_after_fail_stop) > 2:
            psss_failure_list.append(go_after_fail_stop.mean() - go_after_go.mean())

        # ── Lag-1 autocorrelation ──
        if len(valid_go_rts) > 10:
            lag1_list.append(np.corrcoef(valid_go_rts[:-1], valid_go_rts[1:])[0, 1])

        # ── Skewness ──
        if len(valid_go_rts) > 10:
            skew_list.append(sp_stats.skew(valid_go_rts))

        # ── Fatigue drift (slope of RT ~ trial number) ──
        go_with_rt = test[is_go & test["rt"].notna()].copy()
        if len(go_with_rt) > 20:
            trial_nums = np.arange(len(go_with_rt))
            slope, _, _, _, _ = sp_stats.linregress(trial_nums, go_with_rt["rt"].values)
            drift_list.append(slope)

    return {
        "pes": pes_list, "psss": psss_list,
        "psss_success": psss_success_list, "psss_failure": psss_failure_list,
        "lag1": lag1_list, "skewness": skew_list, "drift": drift_list,
    }


def sequential_analysis_stroop(runs: list, fmt: str) -> dict:
    """Compute sequential effects for Stroop data."""
    pes_list, lag1_list, skew_cong_list, skew_incong_list, drift_list = [], [], [], [], []
    gratton_list = []  # sequential congruency effect

    for run_name, exp_df, bot_log in runs:
        if fmt == "expfactory_stroop":
            if "trial_id" not in exp_df.columns:
                continue
            test = exp_df[exp_df["trial_id"] == "test_trial"].copy().reset_index(drop=True)
            test["rt"] = pd.to_numeric(test["rt"], errors="coerce")
            test["correct_trial"] = pd.to_numeric(test["correct_trial"], errors="coerce")
            cond_col = "condition"
        elif fmt == "cognitionrun_stroop":
            test = exp_df[(exp_df["text"].notna()) & (exp_df["text"] != "") &
                          (exp_df["colour"].notna()) & (exp_df["colour"] != "")].copy().reset_index(drop=True)
            test["rt"] = pd.to_numeric(test["rt"], errors="coerce")
            key_map = {"red": "r", "green": "g", "blue": "b", "yellow": "y"}
            test["correct_key"] = test["colour"].map(key_map)
            test["correct_trial"] = (test["response"] == test["correct_key"]).astype(float)
            test["condition"] = np.where(test["text"] == test["colour"], "congruent", "incongruent")
            cond_col = "condition"
        else:
            continue

        valid_rts = test[test["rt"].notna()]["rt"].values
        if len(valid_rts) < 10:
            continue

        # ── Post-error slowing ──
        prev_correct = test["correct_trial"].shift(1)
        after_correct = test[(prev_correct == 1) & test["rt"].notna()]["rt"]
        after_error = test[(prev_correct == 0) & test["rt"].notna()]["rt"]
        if len(after_correct) > 5 and len(after_error) > 2:
            pes_list.append(after_error.mean() - after_correct.mean())

        # ── Lag-1 autocorrelation (correct trials only) ──
        correct_rts = test[(test["correct_trial"] == 1) & test["rt"].notna()]["rt"].values
        if len(correct_rts) > 10:
            lag1_list.append(np.corrcoef(correct_rts[:-1], correct_rts[1:])[0, 1])

        # ── Skewness per condition ──
        for cond, target in [("congruent", skew_cong_list), ("incongruent", skew_incong_list)]:
            cond_rts = test[(test[cond_col] == cond) & (test["correct_trial"] == 1) & test["rt"].notna()]["rt"].values
            if len(cond_rts) > 10:
                target.append(sp_stats.skew(cond_rts))

        # ── Fatigue drift ──
        responded = test[test["rt"].notna()].copy()
        if len(responded) > 20:
            trial_nums = np.arange(len(responded))
            slope, _, _, _, _ = sp_stats.linregress(trial_nums, responded["rt"].values)
            drift_list.append(slope)

        # ── Gratton effect (sequential congruency) ──
        prev_cond = test[cond_col].shift(1)
        # Compute Gratton: interference after congruent vs after incongruent
        cI_after_C = test[(prev_cond == "congruent") & (test[cond_col] == "incongruent") & (test["correct_trial"] == 1)]["rt"].dropna()
        cC_after_C = test[(prev_cond == "congruent") & (test[cond_col] == "congruent") & (test["correct_trial"] == 1)]["rt"].dropna()
        cI_after_I = test[(prev_cond == "incongruent") & (test[cond_col] == "incongruent") & (test["correct_trial"] == 1)]["rt"].dropna()
        cC_after_I = test[(prev_cond == "incongruent") & (test[cond_col] == "congruent") & (test["correct_trial"] == 1)]["rt"].dropna()
        if all(len(x) > 3 for x in [cI_after_C, cC_after_C, cI_after_I, cC_after_I]):
            interference_after_C = cI_after_C.mean() - cC_after_C.mean()
            interference_after_I = cI_after_I.mean() - cC_after_I.mean()
            gratton_list.append(interference_after_C - interference_after_I)

    return {
        "pes": pes_list, "lag1": lag1_list,
        "skewness_congruent": skew_cong_list, "skewness_incongruent": skew_incong_list,
        "drift": drift_list, "gratton": gratton_list,
    }


def fmt_dist(values: list, label: str) -> str:
    """Format a distribution summary."""
    if not values:
        return f"  {label}: no data"
    arr = np.array(values)
    return f"  {label}: M={np.mean(arr):.1f}, SD={np.std(arr):.1f}, range=[{np.min(arr):.1f}, {np.max(arr):.1f}], n={len(arr)}"

print("Sequential analysis functions defined.")

Sequential analysis functions defined.


In [163]:
# ── Run Sequential Analyses on Stop Signal Tasks ──────────────────────

print("=" * 70)
print("STOP SIGNAL — Sequential Effects & RT Distribution")
print("=" * 70)

for task_dir, fmt, label in [
    ("stop_signal_task_(rdoc)", "expfactory_stop_signal", "ExpFactory RDoC"),
    ("stop_signal_task_(stop-it)", "stopit", "STOP-IT"),
]:
    runs = load_trial_data(task_dir)
    if not runs:
        print(f"\n{label}: no valid runs found")
        continue

    res = sequential_analysis_stop_signal(runs, fmt)
    print(f"\n── {label} ({len(runs)} runs) ──")
    print(fmt_dist(res["pes"], "Post-error slowing (ms)"))
    print(fmt_dist(res["psss"], "Post-stop slowing — all stop trials (ms)"))
    print(fmt_dist(res["psss_success"], "Post-stop slowing — after successful inhibition (ms)"))
    print(fmt_dist(res["psss_failure"], "Post-stop slowing — after failed inhibition (ms)"))
    print(fmt_dist(res["lag1"], "Lag-1 autocorrelation (r)"))
    print(fmt_dist(res["skewness"], "Go RT skewness"))
    print(fmt_dist(res["drift"], "Fatigue drift (ms/trial)"))

    # Literature benchmarks
    print("\n  Literature benchmarks:")
    print("    PES: ~20-60ms (Dutilh et al., 2012)")
    print("    Post-stop slowing: ~20-40ms (Verbruggen & Logan, 2008)")
    print("    Post-stop (after successful): ~15-30ms  |  (after failed): ~30-70ms")
    print("    Lag-1 autocorr: ~0.1-0.3 (Wagenmakers et al., 2004)")
    print("    RT skewness: ~0.8-2.0 for ex-Gaussian (Heathcote et al., 1991)")
    print("    Fatigue drift: ~0.1-0.5 ms/trial (Langner et al., 2010)")

STOP SIGNAL — Sequential Effects & RT Distribution

── ExpFactory RDoC (153 runs) ──
  Post-error slowing (ms): M=28.3, SD=41.5, range=[-59.4, 127.1], n=153
  Post-stop slowing — all stop trials (ms): M=12.5, SD=33.1, range=[-101.2, 127.8], n=153
  Post-stop slowing — after successful inhibition (ms): M=30.2, SD=43.4, range=[-93.5, 244.5], n=153
  Post-stop slowing — after failed inhibition (ms): M=-4.1, SD=45.7, range=[-137.4, 108.1], n=153
  Lag-1 autocorrelation (r): M=0.3, SD=0.2, range=[-0.1, 0.7], n=153
  Go RT skewness: M=1.2, SD=0.5, range=[0.1, 2.6], n=153
  Fatigue drift (ms/trial): M=-1.0, SD=1.2, range=[-3.7, 1.0], n=153

  Literature benchmarks:
    PES: ~20-60ms (Dutilh et al., 2012)
    Post-stop slowing: ~20-40ms (Verbruggen & Logan, 2008)
    Post-stop (after successful): ~15-30ms  |  (after failed): ~30-70ms
    Lag-1 autocorr: ~0.1-0.3 (Wagenmakers et al., 2004)
    RT skewness: ~0.8-2.0 for ex-Gaussian (Heathcote et al., 1991)
    Fatigue drift: ~0.1-0.5 ms/trial (L

In [164]:
# ── Run Sequential Analyses on Stroop Tasks ───────────────────────────

print("=" * 70)
print("STROOP — Sequential Effects & RT Distribution")
print("=" * 70)

for task_dir, fmt, label in [
    ("stroop_color-word_task_(rdoc)", "expfactory_stroop", "ExpFactory RDoC"),
    ("stroop_color-word_task", "cognitionrun_stroop", "Cognition.run"),
]:
    runs = load_trial_data(task_dir)
    if not runs:
        print(f"\n{label}: no valid runs found")
        continue

    res = sequential_analysis_stroop(runs, fmt)
    print(f"\n── {label} ({len(runs)} runs) ──")
    print(fmt_dist(res["pes"], "Post-error slowing (ms)"))
    print(fmt_dist(res["lag1"], "Lag-1 autocorrelation (r)"))
    print(fmt_dist(res["skewness_congruent"], "RT skewness — congruent"))
    print(fmt_dist(res["skewness_incongruent"], "RT skewness — incongruent"))
    print(fmt_dist(res["drift"], "Fatigue drift (ms/trial)"))
    print(fmt_dist(res["gratton"], "Gratton effect (ms)"))

    # Literature benchmarks
    print("\n  Literature benchmarks:")
    print("    PES: ~20-60ms (Dutilh et al., 2012)")
    print("    Lag-1 autocorr: ~0.1-0.3 (Wagenmakers et al., 2004)")
    print("    RT skewness: ~0.8-2.0 for ex-Gaussian (Heathcote et al., 1991)")
    print("    Gratton effect: ~20-50ms in manual Stroop (Kerns et al., 2004)")
    print("    NOTE: Gratton is NOT modeled — expect ~0ms")

STROOP — Sequential Effects & RT Distribution

ExpFactory RDoC: no valid runs found

── Cognition.run (141 runs) ──
  Post-error slowing (ms): M=-258.8, SD=226.5, range=[-656.6, -7.3], n=5
  Lag-1 autocorrelation (r): M=0.1, SD=0.3, range=[-0.5, 0.9], n=141
  RT skewness — congruent: M=1.2, SD=0.8, range=[0.3, 2.4], n=6
  RT skewness — incongruent: M=1.5, SD=0.7, range=[0.4, 2.5], n=5
  Fatigue drift (ms/trial): no data
  Gratton effect (ms): no data

  Literature benchmarks:
    PES: ~20-60ms (Dutilh et al., 2012)
    Lag-1 autocorr: ~0.1-0.3 (Wagenmakers et al., 2004)
    RT skewness: ~0.8-2.0 for ex-Gaussian (Heathcote et al., 1991)
    Gratton effect: ~20-50ms in manual Stroop (Kerns et al., 2004)
    NOTE: Gratton is NOT modeled — expect ~0ms
